Load the model

In [1]:
import torch
from transformers import CLIPModel, CLIPProcessor

# Select GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Load pretrained CLIP ViT-B/32
model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

# Move model to device
model = model.to(device)
model.eval()

# Freeze CLIP parameters
for param in model.parameters():
    param.requires_grad = False

print("CLIP model loaded successfully!")
print("Processor loaded successfully!")
print("All parameters frozen:",
      all(not p.requires_grad for p in model.parameters()))


Using device: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model loaded successfully!
Processor loaded successfully!
All parameters frozen: True


Generate and Save CU Artwork Embeddings

In [2]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import torch

# --------------------------------------
# 1. Dataset paths
# --------------------------------------

PROJECT_ROOT = Path("/home/jovyan/AI_summer_shool_Aesha")

METADATA_PATH = (
    PROJECT_ROOT / "processed_gallery/gallery_metadata.csv"
)

IMAGES_ROOT = (
    PROJECT_ROOT / "processed_gallery/images"
)

EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------
# 2. Load gallery metadata
# --------------------------------------

metadata_df = pd.read_csv(METADATA_PATH)

print("Total CU artworks:", len(metadata_df))

# --------------------------------------
# 3. Generate embeddings in batches
# --------------------------------------

batch_size = 16

all_embeddings = []
all_artwork_ids = []

for start in tqdm(
    range(0, len(metadata_df), batch_size),
    desc="Generating CLIP embeddings"
):
    batch = metadata_df.iloc[start:start + batch_size]

    images = []

    for row in batch.itertuples():
        image_path = (
            IMAGES_ROOT
            / f"floor_{row.floor}"
            / row.filename
        )

        with Image.open(image_path) as img:
            images.append(img.convert("RGB"))

    # Preprocess images for CLIP
    inputs = processor(
        images=images,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Extract pretrained image features
    with torch.inference_mode():
        features = model.get_image_features(
            pixel_values=inputs["pixel_values"]
        )

        # Compatible with Transformers versions that
        # return a ModelOutput containing pooler_output
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output

        # L2 normalization
        features = torch.nn.functional.normalize(
            features, p=2, dim=-1
        )

    all_embeddings.append(features.cpu().numpy())
    all_artwork_ids.extend(batch["artwork_id"].tolist())

# --------------------------------------
# 4. Combine embeddings
# --------------------------------------

embeddings = np.vstack(all_embeddings).astype("float32")
artwork_ids = np.array(all_artwork_ids)

print("\nEmbeddings shape:", embeddings.shape)
print("Artwork IDs shape:", artwork_ids.shape)

# --------------------------------------
# 5. Save for future retrieval
# --------------------------------------

save_path = EMBEDDINGS_DIR / "cu_clip_baseline.npz"

np.savez_compressed(
    save_path,
    embeddings=embeddings,
    artwork_ids=artwork_ids
)

print("\nEmbeddings saved successfully!")
print("Saved at:", save_path)

# --------------------------------------
# 6. Basic verification
# --------------------------------------

assert embeddings.shape == (len(metadata_df), 512)
assert len(artwork_ids) == len(metadata_df)
assert np.isfinite(embeddings).all()

print("All embedding checks passed!")


Total CU artworks: 189


Generating CLIP embeddings:   0%|          | 0/12 [00:00<?, ?it/s]


Embeddings shape: (189, 512)
Artwork IDs shape: (189,)

Embeddings saved successfully!
Saved at: /home/jovyan/AI_summer_shool_Aesha/embeddings/cu_clip_baseline.npz
All embedding checks passed!


Retrieve Top 5 Similar CU Artworks Using CLIP

In [3]:

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ----------------------------------
# 1. Load saved gallery embeddings
# ----------------------------------

PROJECT_ROOT = Path("/home/jovyan/AI_summer_shool_Aesha")

embedding_path = PROJECT_ROOT / "embeddings/cu_clip_baseline.npz"
metadata_path = PROJECT_ROOT / "processed_gallery/gallery_metadata.csv"
images_root = PROJECT_ROOT / "processed_gallery/images"

saved = np.load(embedding_path)

gallery_embeddings = saved["embeddings"]
gallery_ids = saved["artwork_ids"]

metadata = pd.read_csv(metadata_path)
metadata = metadata.set_index("artwork_id")

assert gallery_embeddings.shape == (189, 512)
assert len(gallery_ids) == 189

print("Gallery embeddings loaded:", gallery_embeddings.shape)

# ----------------------------------
# 2. Define top-five retrieval
# ----------------------------------

def retrieve_similar_artworks(query_path, top_k=5):

    # Load query image
    with Image.open(query_path) as img:
        query_image = img.convert("RGB")

    # CLIP preprocessing
    inputs = processor(
        images=query_image,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(device)

    # Generate query embedding
    with torch.inference_mode():
        query_features = model.get_image_features(
            pixel_values=pixel_values
        )

        if not isinstance(query_features, torch.Tensor):
            query_features = query_features.pooler_output

        query_features = F.normalize(
            query_features, p=2, dim=-1
        )

    query_embedding = query_features.cpu().numpy()[0]

    # Cosine similarity
    # Both query and gallery embeddings are L2-normalized
    similarity_scores = gallery_embeddings @ query_embedding

    # Get indices of highest similarity
    top_indices = np.argsort(similarity_scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        artwork_id = str(gallery_ids[index])
        artwork = metadata.loc[artwork_id]

        results.append({
            "rank": rank,
            "artwork_id": artwork_id,
            "filename": artwork["filename"],
            "floor": int(artwork["floor"]),
            "similarity": float(similarity_scores[index])
        })

    return query_image, pd.DataFrame(results)

# ----------------------------------
# 3. Select a test query
# ----------------------------------

# Replace with an external photograph available on your GPU server.
# Do not use an identical CU gallery image as the query.

QUERY_PATH = PROJECT_ROOT / "queries/test_photo.jpg"

if not QUERY_PATH.is_file():
    print("Place a test photograph at:", QUERY_PATH)
else:
    query_image, results_df = retrieve_similar_artworks(
        QUERY_PATH,
        top_k=5
    )

    display(results_df)

    # ----------------------------------
    # 4. Display query and top 5 matches
    # ----------------------------------

    fig, axes = plt.subplots(1, 6, figsize=(20, 5))

    axes[0].imshow(query_image)
    axes[0].set_title("Query Image")
    axes[0].axis("off")

    for ax, row in zip(axes[1:], results_df.itertuples()):
        artwork_path = (
            images_root
            / f"floor_{row.floor}"
            / row.filename
        )

        with Image.open(artwork_path) as img:
            ax.imshow(img.convert("RGB"))

        ax.set_title(
            f"Rank {row.rank}\n"
            f"{row.artwork_id}\n"
            f"Floor {row.floor}\n"
            f"Similarity: {row.similarity:.3f}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()


Gallery embeddings loaded: (189, 512)
Place a test photograph at: /home/jovyan/AI_summer_shool_Aesha/queries/test_photo.jpg


 Zero-Shot Painting Attribute Recognition Using CLIP

In [4]:

import io
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets

from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output

# =====================================================
# 1. REQUIREMENTS
# =====================================================

# These must already exist from your earlier cells:
# model, processor, device, images_root
# retrieve_similar_artworks()

assert "model" in globals(), "Run the CLIP initialization cell."
assert "processor" in globals(), "Run the processor initialization cell."
assert "retrieve_similar_artworks" in globals(), "Run the retrieval cell."

model.eval()

PROJECT_ROOT = Path("/home/jovyan/AI_summer_shool_Aesha")
QUERY_DIR = PROJECT_ROOT / "queries"
QUERY_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. ATTRIBUTE VOCABULARY
# =====================================================

ATTRIBUTES = {
    "category": [
        "cityscape", "landscape", "portrait",
        "still life", "abstract art", "architecture",
        "nature scene", "seascape", "animal painting",
        "figurative art"
    ],

    "objects": [
        "buildings", "houses", "windows", "roads",
        "streets", "trees", "flowers", "mountains",
        "water", "rivers", "sky", "clouds",
        "people", "animals", "cars", "bridges",
        "boats", "grass", "snow", "sun", "moon"
    ],

    "patterns": [
        "repeated geometric shapes",
        "repeated vertical lines",
        "repeated horizontal lines",
        "curved flowing lines",
        "strong linear perspective",
        "symmetrical composition",
        "irregular abstract shapes",
        "dense detailed composition",
        "minimalist composition",
        "contrasting color blocks"
    ],

    "season": [
        "spring", "summer", "autumn", "winter"
    ],

    "time": [
        "daytime", "sunrise", "sunset", "nighttime"
    ],

    "mood": [
        "calm", "cheerful", "peaceful",
        "energetic", "melancholic",
        "mysterious", "dramatic", "tense"
    ],

    "style": [
        "realism", "impressionism",
        "expressionism", "abstract art",
        "cubism", "surrealism",
        "pop art", "watercolor illustration",
        "folk art", "modern art"
    ]
}

# =====================================================
# 3. CLIP TEXT FEATURES (CACHED)
# =====================================================

text_cache = {}

def get_text_embeddings(prompts):

    key = tuple(prompts)

    if key not in text_cache:

        inputs = processor(
            text=list(prompts),
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.inference_mode():
            features = model.get_text_features(**inputs)

            if not isinstance(features, torch.Tensor):
                features = features.pooler_output

            features = F.normalize(features, dim=-1)

        text_cache[key] = features.cpu()

    return text_cache[key]


# =====================================================
# 4. EXTRACT IMAGE FEATURES
# =====================================================

def get_image_embedding(image):

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    with torch.inference_mode():

        features = model.get_image_features(
            pixel_values=inputs["pixel_values"].to(device)
        )

        if not isinstance(features, torch.Tensor):
            features = features.pooler_output

        features = F.normalize(features, dim=-1)

    return features.cpu()


# =====================================================
# 5. ZERO-SHOT ATTRIBUTE PREDICTION
# =====================================================

def predict_single(embedding, labels, template, cautious=False):

    prompts = [template.format(label) for label in labels]

    text_features = get_text_embeddings(prompts)

    scores = (embedding @ text_features.T).flatten().numpy()

    order = np.argsort(scores)[::-1]

    best = int(order[0])

    # Ambiguous categories can be left unspecified.
    if cautious and len(order) > 1:
        margin = float(scores[order[0]] - scores[order[1]])

        if margin < 0.015:
            return "Unclear"

    return labels[best].title()


def predict_multi(embedding, labels, max_tags=5):

    predictions = []

    for label in labels:

        prompts = [
            f"An image containing {label}.",
            f"An image without {label}."
        ]

        text_features = get_text_embeddings(prompts)

        scores = (
            embedding @ text_features.T
        ).flatten().numpy()

        margin = float(scores[0] - scores[1])

        # Heuristic zero-shot decision, not calibrated.
        if margin > 0.015:
            predictions.append((label, margin))

    predictions.sort(key=lambda item: item[1], reverse=True)

    return [label for label, _ in predictions[:max_tags]]


# =====================================================
# 6. PIXEL-BASED MAIN COLORS
# =====================================================

COLOR_PALETTE = {
    "Red": (210, 55, 55),
    "Orange": (235, 135, 45),
    "Yellow": (235, 215, 65),
    "Green": (60, 150, 75),
    "Blue": (55, 115, 205),
    "Purple": (140, 80, 175),
    "Pink": (230, 135, 175),
    "Brown": (125, 80, 50),
    "Black": (25, 25, 25),
    "White": (240, 240, 240),
    "Gray": (140, 140, 140),
    "Beige": (210, 190, 155)
}

def analyze_colors(image):

    small = image.convert("RGB").resize((100, 100))

    pixels = np.asarray(small, dtype=np.float32).reshape(-1, 3)

    names = list(COLOR_PALETTE.keys())
    reference = np.array(
        list(COLOR_PALETTE.values()),
        dtype=np.float32
    )

    distances = np.sum(
        (pixels[:, None, :] - reference[None, :, :]) ** 2,
        axis=2
    )

    nearest = np.argmin(distances, axis=1)

    counts = np.bincount(
        nearest,
        minlength=len(names)
    )

    ranked = np.argsort(counts)[::-1]

    dominant = [names[i] for i in ranked[:3]]

    # Approximate color temperature from colored pixels
    import colorsys

    sampled = pixels[::10] / 255.0

    warm = 0
    cool = 0

    for r, g, b in sampled:

        h, s, v = colorsys.rgb_to_hsv(r, g, b)

        if s < 0.18 or v < 0.15:
            continue

        degrees = h * 360

        if degrees < 75 or degrees >= 330:
            warm += 1
        elif 75 <= degrees < 270:
            cool += 1

    total = warm + cool

    if total < 50:
        temperature = "Neutral / unclear"
    else:
        warm_ratio = warm / total

        if warm_ratio >= 0.60:
            temperature = "Mostly warm"
        elif warm_ratio <= 0.40:
            temperature = "Mostly cool"
        else:
            temperature = "Mixed"

    return dominant, temperature


# =====================================================
# 7. COMBINED IMAGE ATTRIBUTE ANALYSIS
# =====================================================

def analyze_artwork(image):

    image = image.convert("RGB")
    embedding = get_image_embedding(image)

    category = predict_single(
        embedding,
        ATTRIBUTES["category"],
        "An image depicting {}."
    )

    objects = predict_multi(
        embedding,
        ATTRIBUTES["objects"],
        max_tags=6
    )

    patterns = predict_multi(
        embedding,
        ATTRIBUTES["patterns"],
        max_tags=3
    )

    colors, temperature = analyze_colors(image)

    season = predict_single(
        embedding,
        ATTRIBUTES["season"],
        "A scene depicting {}.",
        cautious=True
    )

    time_of_day = predict_single(
        embedding,
        ATTRIBUTES["time"],
        "A scene during {}.",
        cautious=True
    )

    mood = predict_single(
        embedding,
        ATTRIBUTES["mood"],
        "An image with a {} mood.",
        cautious=True
    )

    style = predict_single(
        embedding,
        ATTRIBUTES["style"],
        "An artwork in the style of {}.",
        cautious=True
    )

    # Avoid asserting season/time for nonrepresentational art.
    if category in ["Abstract Art", "Still Life", "Portrait"]:
        season = "Not determined"
        time_of_day = "Not determined"

    return {
        "Category": category,
        "Main colors": colors,
        "Color temperature": temperature,
        "Visible objects": objects,
        "Patterns": patterns,
        "Season": season,
        "Time of day": time_of_day,
        "Possible mood": mood,
        "Apparent style": style
    }


# =====================================================
# 8. FIND SHARED FEATURES
# =====================================================

def shared_features(query_attrs, artwork_attrs):

    shared = []

    for field in ["Main colors", "Visible objects", "Patterns"]:

        common = (
            set(query_attrs[field])
            & set(artwork_attrs[field])
        )

        if common:
            shared.append(
                f"{field}: {', '.join(sorted(common))}"
            )

    return "; ".join(shared) if shared else "No shared tags detected"


def format_tags(values):
    if isinstance(values, list):
        return ", ".join(values) if values else "None confidently selected"
    return str(values)


# =====================================================
# 9. RETRIEVE AND DISPLAY EVERYTHING
# =====================================================

def run_artmatch(query_path):

    # Retrieve 5 visually similar CU artworks
    query_image, results_df = retrieve_similar_artworks(
        query_path,
        top_k=5
    )

    # Analyze query
    print("Analyzing uploaded photograph...")
    query_attrs = analyze_artwork(query_image)

    image_list = [query_image]
    titles = ["QUERY IMAGE"]

    detailed_results = []

    # Analyze all retrieved artworks independently
    for row in results_df.itertuples():

        artwork_path = (
            images_root
            / f"floor_{row.floor}"
            / row.filename
        )

        with Image.open(artwork_path) as img:
            artwork_image = img.convert("RGB")

        attrs = analyze_artwork(artwork_image)

        image_list.append(artwork_image)

        titles.append(
            f"Rank {row.rank} | {row.artwork_id}\n"
            f"Floor {row.floor} | Similarity {row.similarity:.3f}"
        )

        detailed_results.append({
            "Artwork ID": row.artwork_id,
            "Floor": row.floor,
            "Similarity": round(row.similarity, 3),
            **{
                key: format_tags(value)
                for key, value in attrs.items()
            },
            "Shared with query": shared_features(
                query_attrs, attrs
            )
        })

    # Display query + retrieved artwork images
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))

    for ax, image, title in zip(
        axes.flat, image_list, titles
    ):
        ax.imshow(image)
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    # Display query attributes
    print("\nQUERY IMAGE ATTRIBUTES")
    print("-" * 50)

    for key, value in query_attrs.items():
        print(f"{key}: {format_tags(value)}")

    # Display retrieved artwork descriptions
    print("\nTOP 5 RETRIEVED ARTWORKS — ATTRIBUTE ANALYSIS")
    print("-" * 50)

    details_df = pd.DataFrame(detailed_results)

    for record in detailed_results:
        print(
            f"\n{record['Artwork ID']} | "
            f"Floor {record['Floor']} | "
            f"Similarity {record['Similarity']:.3f}"
        )

        for key, value in record.items():
            if key not in ["Artwork ID", "Floor", "Similarity"]:
                print(f"  {key}: {value}")

    return results_df, details_df


# =====================================================
# 10. INTERACTIVE UPLOAD WIDGET
# =====================================================

upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image"
)

analysis_output = widgets.Output()

def handle_upload(change):

    if not upload_widget.value:
        return

    with analysis_output:
        clear_output(wait=True)

        try:
            uploaded_file = upload_widget.value[0]

            filename = Path(uploaded_file["name"]).name
            content = bytes(uploaded_file["content"])

            # Validate actual image content
            with Image.open(io.BytesIO(content)) as img:
                img.verify()

            query_path = QUERY_DIR / filename
            query_path.write_bytes(content)

            print("Uploaded:", filename)

            results_df, details_df = run_artmatch(
                query_path
            )

        except Exception as error:
            print("Processing failed:", str(error))

upload_widget.observe(
    handle_upload,
    names="value"
)

display(
    widgets.HTML(
        "<h3>ArtMatch Campus - Visual Artwork Discovery</h3>"
        "<p>Upload a photograph to discover related CU artworks "
        "and their predicted visual attributes.</p>"
    ),
    upload_widget,
    analysis_output
)


HTML(value='<h3>ArtMatch Campus — Visual Artwork Discovery</h3><p>Upload a photograph to discover related CU a…

FileUpload(value=(), accept='image/*', description='Upload Image')

Output()